In [1]:
import polars as pl
from pathlib import Path
import plotly.graph_objs as go
import builtins
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as f
import findspark
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import os

# Yellow cabs

## **Data Loading**

In [2]:
projectRoot = Path.cwd().parents[1]
dataPath = Path(projectRoot, 
                "data/raw/tlc_trip_record/yellow") # Replace with the path of your data
baseName = "yellow_tripdata_"


In [3]:
data = pl.DataFrame().lazy()
years = range(2022,2026,1)
months = [ f"{m:02d}" for m in range(1,13,1)]
dfs = []
for year in years:
    for month in months:
        filePath = Path(dataPath, baseName + f"{year}-{month}.parquet") 
        if (not filePath.exists() or not filePath.is_file()):
            continue
        try:
            d = pl.scan_parquet(filePath).with_columns([(pl.col(pl.Datetime).dt.cast_time_unit("us")),(pl.col("VendorID").cast(pl.Int64))])
        except FileNotFoundError:
            print(f"File {filePath} doesn't exist")
            continue
        dfs.append(d.rename({"Airport_fee": "airport_fee"}) if "Airport_fee" in d.columns else d)
data = pl.concat(dfs, how = "diagonal_relaxed").with_columns([
                ((pl.col('tpep_dropoff_datetime') - pl.col('tpep_pickup_datetime'))
                .dt.total_seconds() / 60).alias('duration_minutes')
            ])



/tmp/ipykernel_4665/474318279.py:15: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  dfs.append(d.rename({"Airport_fee": "airport_fee"}) if "Airport_fee" in d.columns else d)


In [4]:
info = data.collect_schema()

In [5]:
info

Schema([('VendorID', Int64),
        ('tpep_pickup_datetime', Datetime(time_unit='us', time_zone=None)),
        ('tpep_dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
        ('passenger_count', Float64),
        ('trip_distance', Float64),
        ('RatecodeID', Float64),
        ('store_and_fwd_flag', String),
        ('PULocationID', Int64),
        ('DOLocationID', Int64),
        ('payment_type', Int64),
        ('fare_amount', Float64),
        ('extra', Float64),
        ('mta_tax', Float64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('improvement_surcharge', Float64),
        ('total_amount', Float64),
        ('congestion_surcharge', Float64),
        ('airport_fee', Float64),
        ('cbd_congestion_fee', Float64),
        ('duration_minutes', Float64)])

### zones id

In [6]:
dataPathtaxi_zone = Path(projectRoot, 
                "data/raw/taxi_zones") # Replace with the path of your data
taxiFileName = "taxi_zone_lookup.csv"
zone_path = Path(dataPathtaxi_zone, taxiFileName) 

df_zones = pl.scan_csv(zone_path)

In [7]:
df_zones.show(10)

LocationID,Borough,Zone,service_zone
i64,str,str,str
1,"""EWR""","""Newark Airport""","""EWR"""
2,"""Queens""","""Jamaica Bay""","""Boro Zone"""
3,"""Bronx""","""Allerton/Pelham Gardens""","""Boro Zone"""
4,"""Manhattan""","""Alphabet City""","""Yellow Zone"""
5,"""Staten Island""","""Arden Heights""","""Boro Zone"""
6,"""Staten Island""","""Arrochar/Fort Wadsworth""","""Boro Zone"""
7,"""Queens""","""Astoria""","""Boro Zone"""
8,"""Queens""","""Astoria Park""","""Boro Zone"""
9,"""Queens""","""Auburndale""","""Boro Zone"""


In [8]:
df_zones = df_zones.drop("service_zone")

In [9]:
df_zones.collect_schema()

Schema([('LocationID', Int64), ('Borough', String), ('Zone', String)])

## Columns selection
- VendorID: indicates the technology provider that collected the information
    - 1 = Creative Mobile Technologies, LLC
    - 2 = Curb Mobility, LLC
    - 6 = Myle Technologies Inc
    - 7 = Helix
    - 5 = ?? No information available

- tpep_pickup_datetime: date and time when the taximeter is activated

- tpep_dropoff_datetime: date and time when the taximeter is deactivated (the trip ends)

- trip_distance: distance calculated by the taximeter, in miles

- PULocationID: ID of the zone where the taximeter is activated

- DOLocationID: ID of the zone where the taximeter is deactivated

- payment_type: numeric code
    - 0 = Flex Fare trip
    - 1 = Credit card
    - 2 = Cash
    - 3 = No charge
    - 4 = Dispute
    - 5 = Unknown
    - 6 = Voided trip

- tip_amount: the tip (cash tips are not included)

- tolls_amount: total amount of tolls paid during the trip

- total_amount: the amount paid by the passenger, including the tip that was not given in cash


In [10]:
col_select = ["VendorID", "tpep_pickup_datetime","tpep_dropoff_datetime","trip_distance", 
              "PULocationID", "DOLocationID", "payment_type", "tip_amount", "tolls_amount", "total_amount", "duration_minutes"]

In [11]:

data = data.select(col_select)

## Data preprocessing

In [12]:
num_rows = data.select(pl.len()).collect().item()
print("Número de filas:",num_rows)
print("Número de columnas:",len(col_select))

Número de filas: 163553640
Número de columnas: 11


¿Are there null values?

In [13]:
nulls_count = data.select(pl.all().null_count()).collect(engine="streaming")

In [14]:
nulls_count

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,PULocationID,DOLocationID,payment_type,tip_amount,tolls_amount,total_amount,duration_minutes
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0


¿Are there wrong values?

In [15]:
tpep_wrong_rows = (
    data.select(
        (pl.col("tpep_pickup_datetime") > pl.col("tpep_dropoff_datetime")).sum().alias("tpep_wrong_rows")
    )
)

In [16]:
tpep_wrong_rows.collect(engine = "streaming")

tpep_wrong_rows
u32
19896


In [17]:
# We show some data. As can be seen, in the first row it says that the trip starts on January 24 and ends on the 22nd; two days earlier. In the second
# row it indicates that after traveling 10 miles, the destination has been reached several seconds before starting the trip. These erroneous data will 
# be removed later on

data.filter(
    pl.col("tpep_dropoff_datetime") < pl.col("tpep_pickup_datetime")
).select(
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance"
).head(2).collect()


tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance
datetime[μs],datetime[μs],f64
2022-01-24 15:23:01,2022-01-22 06:00:37,4.7
2022-01-01 01:01:54,2022-01-01 01:01:36,10.72


In [18]:
(
    data.select(
        (pl.col("trip_distance")<0).sum().alias("tpep_wrong_rows")
    )
).collect()

tpep_wrong_rows
u32
0


In [19]:
# We observe that there are trips where the distance has been 0 miles. We take it as an error, it also does not provide additional geographic information.

(
    data.select(
        (pl.col("trip_distance")<=0).sum().alias("tpep_wrong_rows")
    )
).collect()

tpep_wrong_rows
u32
3374123


In [20]:
# The initial fare in recent years does not go below 2.5 € so we consider any data that costs less than that to be erroneous
(
    data.select(
        (pl.col("total_amount")<2.5).sum().alias("tpep_wrong_rows")
    )
).collect()

tpep_wrong_rows
u32
2701221


In [21]:
(
    data.select(
        (pl.col("tolls_amount")<0).sum().alias("tpep_wrong_rows")
    )
).collect()

tpep_wrong_rows
u32
153683


In [22]:
(
    data.select(
        (pl.col("tip_amount")<0).sum().alias("tpep_wrong_rows")
    )
).collect()

tpep_wrong_rows
u32
8151


¿Any outliers?

In [23]:
def obtain_outliers(c):
    q = data.select([
        pl.col(c).quantile(0.1).alias("q1"), pl.col(c).quantile(0.9).alias("q3")
    ]).collect(engine = "streaming")
    q1, q3 = q.row(0)
    IQR = q3 - q1
    lower = q1 - 1.5 * IQR
    upper = q3 + 1.5 * IQR

CLEANING

In [24]:
data_clean = data.filter(
    # Distance (1 - 50 miles)
    (pl.col('trip_distance') > 0) &
    (pl.col('trip_distance') <= 50) &
    
    # Fare (1 - 500 dollars)
    (pl.col('total_amount') >= 2.5) &
    (pl.col('total_amount') <= 500) &
    
    # Duration (1min - 3h)
    (pl.col('duration_minutes') >= 1) &
    (pl.col('duration_minutes') <= 180) &

    # To eliminate data with wrong year
    (pl.col('tpep_pickup_datetime').dt.year().is_between(years[0], years[-1])) &
    (pl.col('tpep_dropoff_datetime').dt.year().is_between(years[0], years[-1])) &

    # positive tips and tolls
    (pl.col('tip_amount') >= 0) & (pl.col('tolls_amount') >= 0) &

    # We eliminate unknown neighborhoods because it does not provide geographic information
    (pl.col("PULocationID")!=264) & 
    (pl.col("PULocationID")!=265) &
    (pl.col("DOLocationID")!=264) &
    (pl.col("DOLocationID")!=265)
)

In [25]:
# Obtain counts WITHOUT loading all the data
clean_rows = data_clean.select(pl.len()).collect(engine = "streaming").item()
print(f"Original rows: {num_rows:,}")
print(f"Rows after cleaning: {clean_rows:,}")
print(f"Rows removed: {num_rows - clean_rows:,}")
print(f"Percentage removed: {(1 - clean_rows/num_rows)*100:.2f}%")
print(f"Percentage kept: {(clean_rows/num_rows)*100:.2f}%")

Original rows: 163,553,640
Rows after cleaning: 154,738,461
Rows removed: 8,815,179
Percentage removed: 5.39%
Percentage kept: 94.61%


## Dataframes combination

Example (with all columns)

<div><style>
.dataframe > thead > tr,
.dataframe > tbody > tr {
  text-align: right;
  white-space: pre-wrap;
}
</style>
<small>shape: (5, 9)</small>
<table border="1" class="dataframe"><thead><tr><th>trip_distance</th><th>PULocationID</th><th>DOLocationID</th><th>Borough</th><th>Zone</th><th>service_zone</th><th>Borough_right</th><th>Zone_right</th><th>service_zone_right</th>
</tr><tr><td>f64</td><td>i64</td><td>i64</td><td>str</td><td>str</td><td>str</td><td>str</td><td>str</td><td>str</td></tr></thead><tbody><tr><td>9.0</td><td>138</td><td>48</td>
<td>&quot;Queens&quot;</td><td>&quot;LaGuardia Airport&quot;</td><td>&quot;Airports&quot;</td><td>&quot;Manhattan&quot;</td><td>&quot;Clinton East&quot;</td><td>&quot;Yellow Zone&quot;</td>
</tr><tr><td>9.0</td><td>138</td><td>100</td><td>&quot;Queens&quot;</td><td>&quot;LaGuardia Airport&quot;</td><td>&quot;Airports&quot;</td><td>&quot;Manhattan&quot;</td><td>&quot;Garment District&quot;</td><td>&quot;Yellow Zone&quot;</td></tr><tr><td>8.49</td><td>138</td><td>100</td><td>&quot;Queens&quot;</td><td>&quot;LaGuardia Airport&quot;</td><td>&quot;Airports&quot;</td><td>&quot;Manhattan&quot;</td><td>&quot;Garment District&quot;</td><td>&quot;Yellow Zone&quot;</td></tr><tr><td>8.45</td><td>138</td><td>100</td><td>&quot;Queens&quot;</td><td>&quot;LaGuardia Airport&quot;</td><td>&quot;Airports&quot;</td><td>&quot;Manhattan&quot;</td><td>&quot;Garment District&quot;</td><td>&quot;Yellow Zone&quot;</td></tr><tr><td>8.11</td><td>138</td><td>37</td><td>&quot;Queens&quot;</td><td>&quot;LaGuardia Airport&quot;</td><td>&quot;Airports&quot;</td><td>&quot;Brooklyn&quot;</td><td>&quot;Bushwick South&quot;</td><td>&quot;Boro Zone&quot;</td></tr></tbody></table></div>

In [26]:
col_df_with_zones = ["VendorID", "tpep_pickup_datetime","tpep_dropoff_datetime","trip_distance", 
              "PULocationID", "DOLocationID", "payment_type", "tip_amount", "tolls_amount", "total_amount",
              "PUBorough", "PUZone", "DOBorough","DOZone"]

In [27]:
#In polars, there is no broadcast for join as in spark
df_with_zones = (data_clean.join(df_zones,left_on="PULocationID", right_on = "LocationID", how = "inner",).
                 join(df_zones,left_on="DOLocationID", right_on = "LocationID", how = "inner"))

In [28]:
df_with_zones.collect_schema()

Schema([('VendorID', Int64),
        ('tpep_pickup_datetime', Datetime(time_unit='us', time_zone=None)),
        ('tpep_dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
        ('trip_distance', Float64),
        ('PULocationID', Int64),
        ('DOLocationID', Int64),
        ('payment_type', Int64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('total_amount', Float64),
        ('duration_minutes', Float64),
        ('Borough', String),
        ('Zone', String),
        ('Borough_right', String),
        ('Zone_right', String)])

In [29]:
df_with_zones = df_with_zones.rename({"Borough":"PUBorough", "Zone": "PUZone", "Borough_right": "DOBorough", "Zone_right":"DOZone"})

## Data analysis

Firstly, we study VendorID. We observe that most of the data has been obtained by crub mobility and that every data provided by Helix were not trustworthy


In [30]:
data.group_by(["VendorID"]).len().sort("len", descending = True).collect()

VendorID,len
i64,u32
2,123198846
1,39786145
7,478715
6,89791
5,143


In [31]:
data_clean.group_by(["VendorID"]).len().sort("len", descending = True).collect(engine = "streaming")

VendorID,len
i64,u32
2,116713442
1,38006979
6,17901
5,139


### By zone


End  of the trip zones

In [32]:
df_max_zones = (data_clean
 .select("PULocationID","DOLocationID")
 .join(df_zones,left_on="PULocationID", right_on = "LocationID", how = "inner",)
 .join(df_zones,left_on="DOLocationID", right_on = "LocationID", how = "inner")
 .rename({"Borough":"PUBorough", "Zone": "PUZone", "Borough_right": "DOBorough", "Zone_right":"DOZone"}))


In [33]:
df_max_zones.group_by(["DOZone", "DOBorough"]).len().sort("len", descending = True).head(10).collect(engine = "streaming")

DOZone,DOBorough,len
str,str,u32
"""Upper East Side North""","""Manhattan""",6738148
"""Upper East Side South""","""Manhattan""",6491308
"""Midtown Center""","""Manhattan""",5833284
"""Times Sq/Theatre District""","""Manhattan""",4747321
"""Murray Hill""","""Manhattan""",4569205
"""Midtown East""","""Manhattan""",4335670
"""Lincoln Square East""","""Manhattan""",4264203
"""Upper West Side South""","""Manhattan""",4258964
"""Lenox Hill West""","""Manhattan""",4060167


In [34]:
df_DOBoroughs = df_max_zones.group_by(["DOBorough"]).len().sort("len", descending = True).collect(engine = "streaming")
df_DOBoroughs

DOBorough,len
str,u32
"""Manhattan""",138341178
"""Queens""",8013718
"""Brooklyn""",6834550
"""Bronx""",1120856
"""EWR""",389660
"""Staten Island""",38499


We observe that almost 90% of taxis go to Manhattan

In [35]:
df_DOBoroughs[0,1]/clean_rows*100

89.4032272946026

Where are most taxis taken?

In [36]:
df_max_zones.group_by(["PUZone", "PUBorough"]).len().sort("len", descending = True).head(10).collect(engine = "streaming")

PUZone,PUBorough,len
str,str,u32
"""Upper East Side South""","""Manhattan""",7274370
"""JFK Airport""","""Queens""",6962747
"""Midtown Center""","""Manhattan""",6930712
"""Upper East Side North""","""Manhattan""",6478237
"""Midtown East""","""Manhattan""",5263720
"""Penn Station/Madison Sq West""","""Manhattan""",5180741
"""Times Sq/Theatre District""","""Manhattan""",5042408
"""Lincoln Square East""","""Manhattan""",4959150
"""LaGuardia Airport""","""Queens""",4627604


In [37]:
df_PUBoroughs = df_max_zones.group_by(["PUBorough"]).len().sort("len", descending = True).collect(engine = "streaming")
df_PUBoroughs

PUBorough,len
str,u32
"""Manhattan""",137937868
"""Queens""",13790923
"""Brooklyn""",2490657
"""Bronx""",509326
"""Staten Island""",8392
"""EWR""",1295



Same as before, in the final destination, most of the yellow taxis (89%) are concentrated and start the journey in Manhattan


In [38]:
df_PUBoroughs[0,1]/clean_rows*100

89.14258750447311

In Queens, more than 80% of the taxis that start the trip depart from the airport (JFK Airport + LaGuardia Airport)


In [39]:
(6962747 + 4627604)/df_PUBoroughs.filter(pl.col("PUBorough")=="Queens").row()[1]

0.8404333052979848

In [40]:
df_common_trip = df_max_zones.group_by(["PUBorough", "DOBorough"]).len().sort("len", descending = True).head(37).collect(engine = "streaming")
df_common_trip

PUBorough,DOBorough,len
str,str,u32
"""Manhattan""","""Manhattan""",129128031
"""Queens""","""Manhattan""",8150518
"""Manhattan""","""Queens""",4582626
"""Manhattan""","""Brooklyn""",3288800
"""Queens""","""Queens""",3087040
…,…,…
"""Bronx""","""EWR""",81
"""Staten Island""","""EWR""",67
"""EWR""","""Brooklyn""",47


In [ ]:
unique_boroughs = pl.concat([df_common_trip["PUBorough"], df_common_trip["DOBorough"]]).unique().sort().to_list()
tam_b = builtins.len(unique_boroughs)
labels_ini = [x for x in unique_boroughs]
labels_end = [x for x in unique_boroughs]
labels = labels_ini + labels_end
INI_dict = {x:i for i,x in enumerate(unique_boroughs)} 
END_dict = {x:tam_b+i for i,x in enumerate(unique_boroughs)} 
source = [INI_dict[i] for i in df_common_trip["PUBorough"].to_list()]
target = [END_dict[i] for i in df_common_trip["DOBorough"].to_list()]



fig = go.Figure(data=[go.Sankey(
    node = dict(
      label = labels,
      pad = 25,
      thickness = 15
    ),
    link = dict(
      source = source, 
      target = target,
      value = df_common_trip["len"].to_list()
  ))])


fig.update_layout(title_text="Flujo de viajes de taxis amarillos", font_size=15, height = 500, width = 700)
fig.show()

In [42]:
def generate_figure(df_filter_grouped, B_source, B_target):
  unique_zones = pl.concat([df_filter_grouped["PUZone"], df_filter_grouped["DOZone"]]).unique().sort().to_list()
  tam_b = builtins.len(unique_zones)
  labels_ini = [x for x in unique_zones]
  labels_end = [x for x in unique_zones]
  labels = labels_ini + labels_end
  INI_dict = {x:i for i,x in enumerate(unique_zones)} 
  END_dict = {x:tam_b+i for i,x in enumerate(unique_zones)} 
  source = [INI_dict[i] for i in df_filter_grouped["PUZone"].to_list()]
  target = [END_dict[i] for i in df_filter_grouped["DOZone"].to_list()]


  fig = go.Figure(data=[go.Sankey(
      node = dict(
        label = labels,
        pad = 25,
        thickness = 15
      ),
      link = dict(
        source = source, 
        target = target,
        value = df_filter_grouped["len"].to_list()
    ))])


  fig.update_layout(title_text=f"Flujo de viajes de taxis amarillos desde {B_source} a {B_target}", font_size=15, height = 900, width = 900)
  fig.show()


def generate_sankey_zones(B_source: str, B_target: str, Bar : str | None = None):
    PU_ids = (df_zones.filter(pl.col("Borough")==B_source).select("LocationID"))
    DO_ids = (df_zones.filter(pl.col("Borough")==B_target).select("LocationID"))

    #To filter, you can use join with how = "semi"
    #semi: Returns rows from the left table that have a match in the right table.
    #https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.join.html
    df_filter_zones = (data_clean
    .select("PULocationID","DOLocationID")
    .join(PU_ids, left_on = "PULocationID", right_on = "LocationID", how = "semi")
    .join(DO_ids, left_on = "DOLocationID", right_on = "LocationID", how = "semi")
    .join(df_zones,left_on="PULocationID", right_on = "LocationID", how = "inner",)
    .join(df_zones,left_on="DOLocationID", right_on = "LocationID", how = "inner")
    .rename({"Borough":"PUBorough", "Zone": "PUZone", "Borough_right": "DOBorough", "Zone_right":"DOZone"})
    .select("PUZone","DOZone"))



    df_filter_grouped = df_filter_zones.group_by(["PUZone", "DOZone"]).len().sort("len", descending = True).head(35).collect(engine = "streaming")
    generate_figure(df_filter_grouped, B_source, B_target)


    if Bar is not None:        
      most_common_PU = df_filter_zones.group_by('PUZone').len().sort("len", descending = True).head(10)
      most_common_DO = df_filter_zones.group_by('DOZone').len().sort("len", descending = True).head(10)
      return most_common_PU.collect(engine="streaming"), most_common_DO.collect(engine=("streaming"))
    else:
      return None, None
    

We study mainly Manhattan

In [ ]:
ini, fin = generate_sankey_zones("Manhattan", "Manhattan", "Df_bar")

In [44]:
ini

PUZone,len
str,u32
"""Upper East Side South""",7037573
"""Midtown Center""",6441382
"""Upper East Side North""",6295816
"""Midtown East""",4887433
"""Penn Station/Madison Sq West""",4876660
"""Lincoln Square East""",4756411
"""Times Sq/Theatre District""",4448331
"""Murray Hill""",4204472
"""Upper West Side South""",4142331


In [45]:
fin

DOZone,len
str,u32
"""Upper East Side North""",6516141
"""Upper East Side South""",6297019
"""Midtown Center""",5440943
"""Murray Hill""",4246372
"""Lincoln Square East""",4070877
"""Times Sq/Theatre District""",4035987
"""Upper West Side South""",4031916
"""Midtown East""",4017102
"""Lenox Hill West""",3861059


In [ ]:
generate_sankey_zones("Manhattan", "Brooklyn")

In [ ]:
generate_sankey_zones("Manhattan", "Bronx")

In [ ]:
generate_sankey_zones("Manhattan", "Staten Island")

In [ ]:
generate_sankey_zones("Manhattan", "Queens")

In [ ]:
generate_sankey_zones("Queens", "Manhattan")

## Geojson Analysis

In [ ]:
"""
#To import geojson files, we can use the json library
#uncomment the following code to see how to import geojson files and access its data

import json

geoFile = Path(dataPathtaxi_zone, "NYC_Taxi_Zones_20260207.geojson")
with open(geoFile) as f:
    gZones = json.load(f)

gZones["features"][0]["properties"]"""

# FHVHV

In [46]:
projectRoot = Path.cwd().parents[1]

CLEAN_DATA = projectRoot / "data" / "clean" / "clean_tlc_trip_record" / "fhvhv"
TABLES_DIR = projectRoot / "data" / "plotTables" / "geographical" / "fhvhv"
os.makedirs(TABLES_DIR, exist_ok=True)


In [47]:
def save_table(df:pd.DataFrame, name:str):
    path = os.path.join(str(TABLES_DIR), f"{name}.csv")
    df.to_csv(path, index=False)

def load_csv(path:Path) -> pd.DataFrame:
    return pd.read_csv(path)

In [ ]:
findspark.init()

spark = (SparkSession.builder
        .master("local[*]")  
        .config("spark.driver.memory", "6g")
        .config("spark.executor.memory", "8g")
        .config("spark.sql.shuffle.partitions", "100")
        .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
        .config("spark.driver.maxResultSize", "1g")
        .config("spark.sql.session.timeZone", "America/New_York")
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate())

In [ ]:
spark.sparkContext

In [51]:
df_fhv = spark.read.parquet(str(CLEAN_DATA))

Clean FHVHV data already contains the burough and the zone name.

In [ ]:
for i in df_fhv.schema:
    print(i)

In [53]:
col_select_2 = ["request_datetime", "pickup_datetime","dropoff_datetime","trip_miles", "trip_time", 
              "pickup_zone", "dropoff_zone", "pickup_borough", "dropoff_borough", "tips", "total_amount"]

In [54]:
df_fhv = df_fhv.select(col_select_2)

In [55]:
rows_fhv = df_fhv.count()
print(f"Number of rows in FHV dataset: {rows_fhv:,}")
cols_fhv = len(df_fhv.columns)
print(f"Number of columns in FHV dataset: {cols_fhv}")

Number of rows in FHV dataset: 196,132,613
Number of columns in FHV dataset: 11


### Pickup analysis

We observe that in comparison with yellow taxis, fhvhv are are not that concentrated in Manhattan, but they are more distributed across the different boroughs.

In [56]:
df_pickup= df_fhv.groupBy("pickup_borough").count().orderBy(f.desc("count")).toPandas()
df_pickup

,pickup_borough,count
0,Manhattan,74855434
1,Brooklyn,53547106
2,Queens,41124890
3,Bronx,23614981
4,Staten Island,2990199
5,EWR,3


In [57]:
for i, _ in df_pickup.iterrows():
    print(f'Borough: {df_pickup.iloc[i,0]}, percentage {df_pickup.iloc[i,1]/rows_fhv*100:.2f}%')

Borough: Manhattan, percentage 38.17%
Borough: Brooklyn, percentage 27.30%
Borough: Queens, percentage 20.97%
Borough: Bronx, percentage 12.04%
Borough: Staten Island, percentage 1.52%
Borough: EWR, percentage 0.00%


In [58]:
df_fhv.groupBy("pickup_zone", "pickup_borough").count().orderBy(f.desc("count")).show(10)

+--------------------+--------------+-------+
|         pickup_zone|pickup_borough|  count|
+--------------------+--------------+-------+
|   LaGuardia Airport|        Queens|3834801|
|         JFK Airport|        Queens|2899843|
| Crown Heights North|      Brooklyn|2578247|
|        East Village|     Manhattan|2478337|
|      Midtown Center|     Manhattan|2314483|
|Times Sq/Theatre ...|     Manhattan|2307540|
|TriBeCa/Civic Center|     Manhattan|2245716|
|      Bushwick South|      Brooklyn|2218239|
|       East New York|      Brooklyn|2162449|
|            Union Sq|     Manhattan|2126251|
+--------------------+--------------+-------+
only showing top 10 rows


### Dropoff analysis

In [59]:
df_dropoff = df_fhv.groupBy("dropoff_borough").count().orderBy(f.desc("count")).toPandas()
df_dropoff

,dropoff_borough,count
0,Manhattan,71398350
1,Brooklyn,53653189
2,Queens,42915593
3,Bronx,23747369
4,Staten Island,3002191
5,EWR,1415921


The same happends in the dropoffs as in the pickups. The borough where more people go to is Manhattan, but still it is only a 36% of all the trips

In [60]:
for i, _ in df_dropoff.iterrows():
    print(f'Borough: {df_dropoff.iloc[i,0]}, percentage {df_dropoff.iloc[i,1]/rows_fhv*100:.2f}%')

Borough: Manhattan, percentage 36.40%
Borough: Brooklyn, percentage 27.36%
Borough: Queens, percentage 21.88%
Borough: Bronx, percentage 12.11%
Borough: Staten Island, percentage 1.53%
Borough: EWR, percentage 0.72%


In [61]:
df_fhv.groupBy("dropoff_zone", "dropoff_borough").count().orderBy(f.desc("count")).show(10)

+--------------------+---------------+-------+
|        dropoff_zone|dropoff_borough|  count|
+--------------------+---------------+-------+
|   LaGuardia Airport|         Queens|4474700|
|         JFK Airport|         Queens|4267727|
| Crown Heights North|       Brooklyn|2755444|
|      Bushwick South|       Brooklyn|2257667|
|      Midtown Center|      Manhattan|2221618|
|        East Chelsea|      Manhattan|2213515|
|Times Sq/Theatre ...|      Manhattan|2199546|
|       East New York|       Brooklyn|2170928|
|        East Village|      Manhattan|2133945|
|West Chelsea/Huds...|      Manhattan|2114720|
+--------------------+---------------+-------+
only showing top 10 rows


## Visualisations

In [62]:
def generate_figure(df_filter_grouped, title = None):
  unique_zones = pd.concat([df_filter_grouped["pickup_zone"], df_filter_grouped["dropoff_zone"]]).unique().tolist()
  tam_b = builtins.len(unique_zones)
  labels_ini = [x for x in unique_zones]
  labels_end = [x for x in unique_zones]
  labels = labels_ini + labels_end
  INI_dict = {x:i for i,x in enumerate(unique_zones)} 
  END_dict = {x:tam_b+i for i,x in enumerate(unique_zones)} 
  source = [INI_dict[i] for i in df_filter_grouped["pickup_zone"].to_list()]
  target = [END_dict[i] for i in df_filter_grouped["dropoff_zone"].to_list()]


  fig = go.Figure(data=[go.Sankey(
      node = dict(
        label = labels,
        pad = 25,
        thickness = 15
      ),
      link = dict(
        source = source, 
        target = target,
        value = df_filter_grouped["count"].to_list()
    ))])

  if title is not None:
    fig.update_layout(title_text=title, font_size=15, height = 900, width = 900)
  else:
    fig.update_layout(title_text=f"Flujo de viajes de fhvhv", font_size=15, height = 900, width = 900)
  fig.show()


def generate_sankey_zones(df, title = None):
    df = (df.select("pickup_zone", "pickup_borough", "dropoff_zone", "dropoff_borough"))
    df_filter_grouped = df.groupBy(["pickup_zone", "dropoff_zone"]).count().sort("count", ascending = False).limit(35).toPandas()
    generate_figure(df_filter_grouped, title)
    

In [ ]:
generate_sankey_zones(df_fhv)

In [ ]:
B_source = "Manhattan"
B_target = "Manhattan"
title = f"Flujo de viajes de fhvhv desde {B_source} a {B_target}"
generate_sankey_zones(df_fhv.filter((f.col("pickup_borough") == B_source) & (f.col("dropoff_borough") == B_target)), title)

In [ ]:
B_source = "Manhattan"
B_target = "Queens"
title = f"Flujo de viajes de fhvhv desde {B_source} a {B_target}"
generate_sankey_zones(df_fhv.filter((f.col("pickup_borough") == B_source) & (f.col("dropoff_borough") == B_target)), title)

In [ ]:
B_source = "Queens"
B_target = "Queens"
title = f"Flujo de viajes de fhvhv desde {B_source} a {B_target}"
generate_sankey_zones(df_fhv.filter((f.col("pickup_borough") == B_source) & (f.col("dropoff_borough") == B_target)), title)

In [ ]:
B_source = "Manhattan"
B_target = "Brooklyn"
title = f"Flujo de viajes de fhvhv desde {B_source} a {B_target}"
generate_sankey_zones(df_fhv.filter((f.col("pickup_borough") == B_source) & (f.col("dropoff_borough") == B_target)), title)

In [ ]:
B_source = "Brooklyn"
B_target = "Brooklyn"
title = f"Flujo de viajes de fhvhv desde {B_source} a {B_target}"
generate_sankey_zones(df_fhv.filter((f.col("pickup_borough") == B_source) & (f.col("dropoff_borough") == B_target)), title)

All boroughs

In [ ]:
df_com = df_fhv.select("pickup_borough", "dropoff_borough").groupBy(["pickup_borough", "dropoff_borough"]).count().sort("count", ascending = False).toPandas()
unique_boroughs = pd.concat([df_com["pickup_borough"], df_com["dropoff_borough"]]).unique().tolist()

tam_b = builtins.len(unique_boroughs)
labels_ini = [x for x in unique_boroughs]
labels_end = [x for x in unique_boroughs]
labels = labels_ini + labels_end
INI_dict = {x:i for i,x in enumerate(unique_boroughs)} 
END_dict = {x:tam_b+i for i,x in enumerate(unique_boroughs)} 
source = [INI_dict[i] for i in df_com["pickup_borough"].to_list()]
target = [END_dict[i] for i in df_com["dropoff_borough"].to_list()]



fig = go.Figure(data=[go.Sankey(
    node = dict(
      label = labels,
      pad = 25,
      thickness = 15
    ),
    link = dict(
      source = source, 
      target = target,
      value = df_com["count"].to_list()
  ))])


fig.update_layout(title_text="Flujo de viajes de fhv en los distintos barrios de NYC", font_size=15, height = 500, width = 700)
fig.show()